# Ambil Dataset Sentimen dari YouTube

Notebook ini menunjukkan cara mengambil komentar dari video YouTube menggunakan YouTube Data API.

## Persiapan
Kamu butuh **API Key** dari Google Cloud Console:
1. Buka https://console.cloud.google.com
2. Buat project baru (atau pakai yang ada)
3. Aktifkan "YouTube Data API v3"
4. Buat API Key di menu "Credentials"
5. Copy API Key-nya, paste ke file **`.env`** (jangan di notebook ini!)

In [1]:
!pip install google-api-python-client pandas python-dotenv transformers torch

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import re
from dotenv import load_dotenv
import os

# Coba load API key dari file .env (untuk lokal)
load_dotenv()
YOUTUBE_API_KEY = os.getenv('YOUTUBE_API_KEY')

# Kalau .env tidak ada (misal di Colab), minta input manual
if not YOUTUBE_API_KEY:
    print('.env tidak ditemukan. Masukkan API key manual:')
    YOUTUBE_API_KEY = input('Paste YouTube API Key: ').strip()

youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
print('Koneksi ke YouTube API berhasil!')

## Ambil Komentar dari Video

Kita bisa ambil komentar dari video manapun menggunakan Video ID.

Video ID ada di URL: `https://www.youtube.com/watch?v=`**`dQw4w9WgXcQ`**

In [3]:
def ambil_komentar_youtube(video_id, max_komentar=200):
    """Ambil komentar dari video YouTube."""
    komentar_list = []
    next_page_token = None
    
    while len(komentar_list) < max_komentar:
        request = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            maxResults=min(100, max_komentar - len(komentar_list)),
            pageToken=next_page_token,
            textFormat='plainText'
        )
        
        response = request.execute()
        
        for item in response['items']:
            comment = item['snippet']['topLevelComment']['snippet']
            komentar_list.append({
                'teks': comment['textDisplay'],
                'author': comment['authorDisplayName'],
                'like_count': comment['likeCount'],
                'published_at': comment['publishedAt'],
            })
        
        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
    
    return pd.DataFrame(komentar_list)

# Contoh: ambil komentar dari video (ganti VIDEO_ID)
VIDEO_ID = 'hTvfn8-l_mk'  # ganti dengan video yang kamu mau

df_komentar = ambil_komentar_youtube(VIDEO_ID, max_komentar=100)
print(f'Total komentar: {len(df_komentar)}')
df_komentar.head(10)

Total komentar: 100


,teks,author,like_count,published_at
0,Selama orang orang disengaja untuk tetap tidak...,@alil7even,0,2026-06-22T04:30:52Z
1,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...,@ediedi2244,0,2026-06-22T04:13:40Z
2,Alhamdulillah sangat bermanfaat untuk yang waras,@NofRizal-n7m,0,2026-06-22T04:12:54Z
3,Klo ga boong bukan kabib,@fuadhasyimhasyim3777,0,2026-06-22T04:00:13Z
4,emang pekerjaan paling enak di Indonesia tuh j...,@chandrapratama8903,0,2026-06-22T03:29:50Z
5,yg bilang si bahar habit hanya orang dongo,@Berachain,0,2026-06-22T02:56:37Z
6,Yg animasi pesawat mundur lucu bgt,@kucingganteng3148,0,2026-06-22T02:47:22Z
7,"""Karomah tidak untuk dipertontonkan"", ini pemb...",@BrahmaDBA,0,2026-06-22T02:21:29Z
8,"Terlalu banyak orang mabuk agama , sehingga ba...",@dariswargono1755,0,2026-06-22T01:42:48Z
9,Memang kebohongan itu merupakan habitatnya dan...,@supardibasriahmad7337,0,2026-06-22T01:22:45Z


## Ambil Komentar dari Beberapa Video Sekaligus

Bisa juga ambil komentar dari channel tertentu atau beberapa video sekaligus.

In [ ]:
def ambil_video_dari_channel(channel_id, max_video=10):
    """Ambil daftar video terbaru dari channel."""
    request = youtube.search().list(
        part='id',
        channelId=channel_id,
        maxResults=max_video,
        type='video',
        order='date'
    )
    response = request.execute()
    
    video_ids = [item['id']['videoId'] for item in response['items']]
    return video_ids

def ambil_komentar_banyak_video(video_ids, max_per_video=50):
    """Ambil komentar dari beberapa video."""
    semua_komentar = []
    
    for i, vid in enumerate(video_ids):
        print(f'[{i+1}/{len(video_ids)}] Mengambil komentar dari {vid}...')
        df = ambil_komentar_youtube(vid, max_komentar=max_per_video)
        df['video_id'] = vid
        semua_komentar.append(df)
    
    return pd.concat(semua_komentar, ignore_index=True)

# Contoh: ganti CHANNEL_ID dengan channel yang kamu mau
# CHANNEL_ID = 'UCxxxxxxx'  # ada di URL channel
# video_ids = ambil_video_dari_channel(CHANNEL_ID, max_video=5)
# df_semua = ambil_komentar_banyak_video(video_ids, max_per_video=50)
# print(f'Total komentar dari semua video: {len(df_semua)}')

print('Uncomment kode di atas untuk menjalankan')

## Bersihkan Komentar

Komentar YouTube sering mengandung emoji, URL, dan karakter aneh. Kita bersihkan dulu.

In [4]:
def bersihkan_komentar(teks):
    """Bersihkan komentar dari URL, emoji, dan karakter aneh."""
    # Hapus URL
    teks = re.sub(r'http\S+|www\S+', '', teks)
    # Hapus emoji (karakter non-ASCII)
    teks = re.sub(r'[^\x00-\x7F\u0080-\uFFFF]+', '', teks)
    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df_komentar['teks_bersih'] = df_komentar['teks'].apply(bersihkan_komentar)

# Tampilkan perbandingan
df_komentar[['teks', 'teks_bersih']].head(10)

,teks,teks_bersih
0,Selama orang orang disengaja untuk tetap tidak...,Selama orang orang disengaja untuk tetap tidak...
1,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...
2,Alhamdulillah sangat bermanfaat untuk yang waras,Alhamdulillah sangat bermanfaat untuk yang waras
3,Klo ga boong bukan kabib,Klo ga boong bukan kabib
4,emang pekerjaan paling enak di Indonesia tuh j...,emang pekerjaan paling enak di Indonesia tuh j...
5,yg bilang si bahar habit hanya orang dongo,yg bilang si bahar habit hanya orang dongo
6,Yg animasi pesawat mundur lucu bgt,Yg animasi pesawat mundur lucu bgt
7,"""Karomah tidak untuk dipertontonkan"", ini pemb...","""Karomah tidak untuk dipertontonkan"", ini pemb..."
8,"Terlalu banyak orang mabuk agama , sehingga ba...","Terlalu banyak orang mabuk agama , sehingga ba..."
9,Memang kebohongan itu merupakan habitatnya dan...,Memang kebohongan itu merupakan habitatnya dan...


## Simpan ke CSV

Setelah bersih, simpan ke file CSV untuk dipakai di notebook analisis sentimen.

In [6]:
# Simpan ke CSV
df_komentar.to_csv('komentar_youtube.csv', index=False)
print(f'Disimpan ke komentar_youtube.csv ({len(df_komentar)} baris)')

# Download ke komputer lokal
from google.colab import files
files.download('komentar_youtube.csv')

# Cek isi file
pd.read_csv('komentar_youtube.csv').head()

Disimpan ke komentar_youtube.csv (100 baris)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,teks,author,like_count,published_at,teks_bersih
0,Selama orang orang disengaja untuk tetap tidak...,@alil7even,0,2026-06-22T04:30:52Z,Selama orang orang disengaja untuk tetap tidak...
1,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...,@ediedi2244,0,2026-06-22T04:13:40Z,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...
2,Alhamdulillah sangat bermanfaat untuk yang waras,@NofRizal-n7m,0,2026-06-22T04:12:54Z,Alhamdulillah sangat bermanfaat untuk yang waras
3,Klo ga boong bukan kabib,@fuadhasyimhasyim3777,0,2026-06-22T04:00:13Z,Klo ga boong bukan kabib
4,emang pekerjaan paling enak di Indonesia tuh j...,@chandrapratama8903,0,2026-06-22T03:29:50Z,emang pekerjaan paling enak di Indonesia tuh j...


## Auto-Labeling Sentimen pakai Pre-trained Model (HuggingFace)

Komentar YouTube belum punya label sentimen. Kita pakai **model BERT yang sudah dilatih** dari HuggingFace untuk auto-labeling.

Model ini jauh lebih akurat dari keyword matching karena:
- Paham konteks kalimat ("tidak bagus" = negatif, bukan positif)
- Support bahasa Indonesia + Inggris (multilingual)
- Sudah dilatih pakai jutaan data

> Pertama kali jalan agak lama (~1-2 menit) karena download model. Setelah itu cepat.

In [7]:
from transformers import pipeline

# Load pre-trained sentiment model (multilingual, support Indo + Inggris)
# Pertama kali: download model ~500MB, setelah itu cache
print('Loading model... (pertama kali agak lama)')
sentiment_analyzer = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    tokenizer='nlptown/bert-base-multilingual-uncased-sentiment'
)
print('Model siap!')

# Model ini output: 1 star, 2 stars, 3 stars, 4 stars, 5 stars
# Kita convert ke sentimen sederhana
def convert_label(label):
    """Convert '1 star' -> negatif, '5 stars' -> positif, dll."""
    bintang = int(label.split()[0])
    if bintang >= 4:
        return 1     # positif
    elif bintang <= 2:
        return -1    # negatif
    else:
        return 0     # netral

def label_batch(teks_list, batch_size=16):
    """Label banyak teks sekaligus (lebih cepat dari satu-satu)."""
    results = []
    
    for i in range(0, len(teks_list), batch_size):
        batch = teks_list[i:i+batch_size]
        # Truncate teks yang terlalu panjang (max 512 token)
        batch = [t[:500] if len(t) > 500 else t for t in batch]
        
        predictions = sentiment_analyzer(batch)
        
        for pred in predictions:
            results.append({
                'sentimen': convert_label(pred['label']),
                'confidence': pred['score'],
                'raw_label': pred['label']
            })
    
    return results

# Jalankan auto-labeling
print('Labeling komentar...')
label_results = label_batch(df_komentar['teks_bersih'].tolist())

df_komentar['sentimen'] = [r['sentimen'] for r in label_results]
df_komentar['confidence'] = [r['confidence'] for r in label_results]
df_komentar['raw_label'] = [r['raw_label'] for r in label_results]

print(f'\nDistribusi sentimen:')
print(df_komentar['sentimen'].value_counts().sort_index().map({1: 'Positif', -1: 'Negatif', 0: 'Netral'}))
print(f'\nRata-rata confidence: {df_komentar["confidence"].mean():.2%}')
print()

# Tampilkan contoh
df_komentar[['teks_bersih', 'raw_label', 'sentimen', 'confidence']].head(20)

Loading model... (pertama kali agak lama)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model siap!
Labeling komentar...

Distribusi sentimen:
sentimen
-1    NaN
 0    NaN
 1    NaN
Name: count, dtype: object

Rata-rata confidence: 39.11%



,teks_bersih,raw_label,sentimen,confidence
0,Selama orang orang disengaja untuk tetap tidak...,4 stars,1,0.252055
1,BERDOGENG DAN BOHONG SALAH SATU CARA UNTUK MEN...,5 stars,1,0.354975
2,Alhamdulillah sangat bermanfaat untuk yang waras,5 stars,1,0.636686
3,Klo ga boong bukan kabib,1 star,-1,0.317665
4,emang pekerjaan paling enak di Indonesia tuh j...,1 star,-1,0.571825
5,yg bilang si bahar habit hanya orang dongo,1 star,-1,0.390503
6,Yg animasi pesawat mundur lucu bgt,1 star,-1,0.646914
7,"""Karomah tidak untuk dipertontonkan"", ini pemb...",1 star,-1,0.460466
8,"Terlalu banyak orang mabuk agama , sehingga ba...",1 star,-1,0.436363
9,Memang kebohongan itu merupakan habitatnya dan...,3 stars,0,0.241853


## Simpan Dataset Berlabel

Sekarang kita punya dataset komentar YouTube yang sudah dilabeli sentimen.

In [8]:
df_komentar.to_csv('youtube_sentiment_dataset.csv', index=False)
print(f'Dataset tersimpan: youtube_sentiment_dataset.csv')
print(f'Total: {len(df_komentar)} komentar')

# Download ke komputer lokal
from google.colab import files
files.download('youtube_sentiment_dataset.csv')

print('\nFile ini bisa dipakai di notebook analisis sentimen!')

Dataset tersimpan: youtube_sentiment_dataset.csv
Total: 100 komentar


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


File ini bisa dipakai di notebook analisis sentimen!


## Catatan Penting

- **Quota API gratis**: 10.000 unit/hari. Tiap request commentThreads = 1 unit.
- **Maksimal 100 komentar per request**, pakai pagination untuk lebih banyak.
- **Komentar bisa di-disable** di beberapa video -> API akan return error.
- **Bahasa campuran**: komentar YouTube sering campur bahasa (Indo + Inggris + slang).
- **Model BERT multilingual** yang kita pakai support 50+ bahasa, jadi bisa handle komentar campur bahasa.
- **Colab RAM**: model BERT butuh ~2GB RAM. Kalau Colab crash, coba pakai runtime GPU (Runtime > Change runtime type > GPU).